# AutoMode NeurIPS — Driver for GPU 1 (RTX 5090, 32 GB)

**Role**: mid-size models — Qwen2.5-3B-Instruct (Tier 0) and Gemma-3-4B (Tier 1).

See `driver_gpu0_pro6000.ipynb` for detailed notes on structure.

In [1]:
import os
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'
import torch
print(f'torch {torch.__version__} | cuda={torch.cuda.is_available()} | '
      f'device={torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"}')

torch 2.11.0+cu128 | cuda=True | device=NVIDIA GeForce RTX 5090


In [2]:
import subprocess, sys
result = subprocess.run(
    [sys.executable, '-m', 'pytest', 'tests/test_switching.py', 'tests/test_training_invariants.py', '-v'],
    cwd='..', capture_output=True, text=True,
)
print(result.stdout); print(result.stderr)
assert result.returncode == 0, 'Unit tests FAILED. Do not proceed to training.'

============================= test session starts ==============================
platform linux -- Python 3.12.13, pytest-9.0.3, pluggy-1.6.0 -- /venv/main/bin/python
cachedir: .pytest_cache
rootdir: /workspace/Automode/automode_pkg
configfile: pyproject.toml
plugins: anyio-4.13.0
collecting ... collected 34 items

tests/test_switching.py::TestLayerIdentification::test_double_peft_wrap PASSED [  2%]
tests/test_switching.py::TestLayerIdentification::test_get_transformer_layers_peft_wrapped PASSED [  5%]
tests/test_switching.py::TestLayerIdentification::test_get_transformer_layers_plain PASSED [  8%]
tests/test_switching.py::TestLayerIdentification::test_gets_all_layer_ids PASSED [ 11%]
tests/test_switching.py::TestLayerIdentification::test_layer_ids_sorted_numerically_not_alphabetically PASSED [ 14%]
tests/test_switching.py::TestLayerIdentification::test_non_layer_returns_none PASSED [ 17%]
tests/test_switching.py::TestLayerIdentification::test_peft_wrapped_path PASSED [ 20%]
tests/test

In [ ]:
from automode.grid import build_tier_grid, shard_grid_by_gpu, DEFAULT_GPU_ASSIGNMENT

GPU_RANK = 1

full_grid = build_tier_grid(tiers=[0, 1, 4], output_root='runs')
mine = shard_grid_by_gpu(full_grid, gpu_rank=GPU_RANK, assignment=DEFAULT_GPU_ASSIGNMENT)
for c in mine:
    c.device = 'cuda:0'  # CUDA_VISIBLE_DEVICES=1 remaps this to the physical 5090

from collections import Counter
print(f'Total grid: {len(full_grid)} | My slice: {len(mine)}')
print(f'By model: {dict(Counter(c.model_name for c in mine))}')
print(f'By tier:  {dict(Counter(c.tier for c in mine))}')

In [ ]:
# ── Smoke test on Qwen (smaller / faster to fail fast) ────────────────────
from automode.config import preset_automode
from automode.train import run_experiment
from automode.grid import _apply_model_defaults, _apply_track_defaults

smoke = preset_automode(
    u=10, t=10, r=16, alpha=32,
    model_name='Qwen/Qwen2.5-3B-Instruct',
    train_track='gsm8k',
    eval_benchmarks=('gsm8k',),
    seed=42, epochs=1,
    output_root='runs/smoke',
    max_train_samples=32, max_eval_samples=50, tier=-1,
)
smoke = _apply_track_defaults(_apply_model_defaults(smoke))
smoke.device = 'cuda:0'
smoke_result = run_experiment(smoke)
assert smoke_result['status'] == 'ok', f'Smoke: {smoke_result}'
print({k: v for k, v in smoke_result.items() if k != 'config'})

In [ ]:
from automode.grid import run_grid
import time
CSV_PATH = 'runs/gpu1_results.csv'
t0 = time.time()
results = run_grid(mine, csv_path=CSV_PATH, stop_on_error=False, verbose=True)
print(f'\nGPU 1 complete: {len(results)} rows in {(time.time()-t0)/3600:.1f} hr')